In [1]:
import math
import time
import random
import heapq

def timer(func, *args, **kwargs):
    t0 = time.perf_counter_ns()
    res = func(*args, **kwargs)
    t1 = time.perf_counter_ns()
    us = (t1 - t0) / 1000.0

    return res, us

## 1. Closest pair of points in 2D plane

In [2]:
def dist(p1, p2):
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])

def closest_brute_force(pts):
    n = len(pts)
    min_d, pair = float('inf'), None

    for i in range(n):
        for j in range(i + 1, n):
            d = dist(pts[i], pts[j])
            if d < min_d:
                min_d, pair = d, (pts[i], pts[j])

    return min_d, pair

def closest_divide_conquer(pts):
    px = sorted(pts, key= lambda p: p[0])
    py = sorted(pts, key= lambda p: p[1])

    def rec(px, py):
        n = len(px)
        if n <= 3:
            return closest_brute_force(px)

        mid = n // 2
        mid_x = px[mid][0]
        lx, rx = px[:mid], px[mid:]

        set_lx = set(lx)

        ly = [p for p in py if p in set_lx]
        ry = [p for p in py if p not in set_lx]

        (d1, p1) = rec(lx, ly)
        (d2, p2) = rec(rx, ry)
        d, pair = (d1, p1) if d1<d2 else (d2, p2)

        strip = [p for p in py if abs(p[0] - mid_x) < d]
        for i in range(len(strip)):
            for j in range(i+1, min(i + 8, len(strip))):
                sd = dist(strip[i], strip[j])
                if sd < d:
                    d, pair = sd, (strip[i], strip[j])

        return d, pair

    return rec(px, py)

In [3]:
random.seed(42)

print(f"{'N':>6}  |  {'Brute Force (us)':>18}  |  {'Divide & Conquer (us)':>22} | {'Matches':>8}")
print("-" * 62)
for n in [50, 100, 200, 400, 800]:
    pts = [(random.uniform(0, 1000), random.uniform(0,1000)) for _ in range(n)]
    (d_bf, _), t_bf = timer(closest_brute_force,pts)
    (d_dc, _), t_dc = timer(closest_divide_conquer, pts)
    assert math.isclose(d_bf, d_dc), f"Mismatch at N = {n}"
    print(f"{n:>6}  |  {t_bf:>18.2f}  |  {t_dc:>22.2f}  |  {'True':>8}")

     N  |    Brute Force (us)  |   Divide & Conquer (us) |  Matches
--------------------------------------------------------------
    50  |              745.06  |                  434.55  |      True
   100  |             3247.01  |                  925.70  |      True
   200  |            12279.86  |                 2685.30  |      True
   400  |            37725.49  |                 2277.32  |      True
   800  |            93968.28  |                 6745.10  |      True


## Assigment 2

In [4]:
def generate_dataset(n=1_000_000, max_val = 1_500_000):
    return [random.randint(1, max_val) for _ in range(n)]

def resolve_duplicates(arr):
    seen = set()
    duplicates_count = 0
    resolved = []
    occupied = set(arr)

    for x in arr:
        if x not in seen:
            seen.add(x)
            resolved.append(x)
        else:
            duplicates_count += 1
            step = 1
            while True:
                c_up = x + step
                if c_up not in occupied:
                    occupied.add(c_up)
                    resolved.append(c_up)
                    break
                step += 1

    return duplicates_count, resolved

dataset, t_gen = timer(generate_dataset, 1_000_000)
total_duplicates = len(dataset) - len(set(dataset))
print(f"Dataset Size: {len(dataset):,} positive integers")
print(f"Generation Time: {t_gen:,.2f} us ({t_gen / 1000.0:.2f} ms)")
print(f"Duplicates Found in 1M dataset: {total_duplicates:,}")

Dataset Size: 1,000,000 positive integers
Generation Time: 418,983.34 us (418.98 ms)
Duplicates Found in 1M dataset: 270,475


In [5]:
(num_dups, resolved_sample), t_res = timer(resolve_duplicates, dataset)
print(f"Duplicates: {num_dups:,}")
print(f"Resolution Time: {t_res:,.2f} us ({t_res / 1000.0:.2f} ms)")
print(f"Remaining duplicates after resolution: {len(resolved_sample) - len(set(resolved_sample))}")

Duplicates: 270,475
Resolution Time: 745,447.68 us (745.45 ms)
Remaining duplicates after resolution: 0


## 4. Find element at index i if Array were Sorted

In [6]:
def select_heap(arr, i):
    n = len(arr)
    if i < n // 2:
        return heapq.nsmallest(i+1, arr)[-1]
    else:
        return heapq.nlargest(n - i, arr)[-1]


def quickselect(arr, i):
    def partition(a, l, r):
        pivot = a[r]
        p = l
        for j in range(l, r):
            if a[j] <= pivot:
                a[p], a[j] = a[j], a[p]
                p += 1
        a[p], a[r] = a[r], a[p]
        return p

    def select(a, l, r, k):
        if l == r:
            return a[l]
        p = partition(a, l, r)
        if k == p:
            return a[p]
        elif k < p:
            return select(a, l, p-1, k)
        else:
            return select(a, p+1, r, k)

    return select(arr.copy(), 0, len(arr) - 1, i)

def select_sort(arr, i):
    return sorted(arr)[i]


n_arr = 50_000
test_data = [random.randint(1, 1_000_000) for _ in range(n_arr)]
test_indices = [0, 10, n_arr // 2, n_arr - 11, n_arr - 1]

print(f"{'Target i':>10} | {'Heap (µs)':>12} | {'Quickselect (µs)':>18} | {'Sort (µs)':>12} | {'All Match':>10}")
print("-" * 72)

for i in test_indices:
    val_sort, t_sort = timer(select_sort, test_data, i)
    val_heap, t_heap = timer(select_heap, test_data, i)
    val_qs, t_qs = timer(quickselect, test_data, i)
    match = (val_sort == val_heap == val_qs)
    print(f"{i:>10} | {t_heap:>12.2f} | {t_qs:>18.2f} | {t_sort:>12.2f} | {str(match):>10}")


  Target i |    Heap (µs) |   Quickselect (µs) |    Sort (µs) |  All Match
------------------------------------------------------------------------
         0 |       895.37 |           21790.70 |     10474.83 |       True
        10 |      1047.47 |           21973.17 |     10837.95 |       True
     25000 |     91816.80 |           20838.83 |      7960.05 |       True
     49989 |      1028.91 |            7681.88 |      9314.97 |       True
     49999 |       686.85 |            8024.28 |      8159.83 |       True


## 5. Time Complexity

In [7]:
def code1(n):
    x = 0
    for i in range (1, n+1):
        for j in range(1, i+1):
            x += 1
    
    return x

def code2(n):
    x = 0
    j = n
    while j >= 1:
        for i in range(1, j + 1):
            x += 1
        j = j // 2

    return x

def code3(n):
    x = 0
    for i in range (1, n + 1):
        for j in range(1, i + 1):
            for k in range(1, i + 1):
                x +=1

    return x

def code4(n):
    x = 0
    i = n
    while i >= 1:
        for j in range(1, n+1):
            x += 1
        i = i // 2
    return x


print("=== Exact Step Counts vs Theoretical Formulas ===")
for n in [10, 50, 100]:
    c1, _ = timer(code1, n)
    c2, _ = timer(code2, n)
    c3, _ = timer(code3, n)
    c4, _ = timer(code4, n)
    th1 = n * (n + 1) // 2
    th3 = n * (n + 1) * (2* n + 1) // 6
    print(f"n={n:>3} | Code1: {c1:>6} (theo {th1}) | Code2: {c2:>4} | Code3: {c3:>7} (theo {th3}) | Code4: {c4:>5}")


print("\n=== Empirical Execution Times (µs) ===")
print(f"{'n':>5} | {'Code 1 O(n²)':>14} | {'Code 2 O(n)':>13} | {'Code 3 O(n³)':>14} | {'Code 4 O(n log n)':>18}")
print("-" * 74)

for n in [50, 100, 200, 300]:
    _, t1 = timer(code1, n)
    _, t2 = timer(code2, n)
    _, t3 = timer(code3, n)
    _, t4 = timer(code4, n)
    print(f"{n:>5} | {t1:>14.2f} | {t2:>13.2f} | {t3:>14.2f} | {t4:>18.2f}")


=== Exact Step Counts vs Theoretical Formulas ===
n= 10 | Code1:     55 (theo 55) | Code2:   18 | Code3:     385 (theo 385) | Code4:    40
n= 50 | Code1:   1275 (theo 1275) | Code2:   97 | Code3:   42925 (theo 42925) | Code4:   300
n=100 | Code1:   5050 (theo 5050) | Code2:  197 | Code3:  338350 (theo 338350) | Code4:   700

=== Empirical Execution Times (µs) ===
    n |   Code 1 O(n²) |   Code 2 O(n) |   Code 3 O(n³) |  Code 4 O(n log n)
--------------------------------------------------------------------------
   50 |          42.65 |          4.05 |        1452.41 |               7.06
  100 |         159.12 |          4.90 |       14903.37 |              31.45
  200 |        1365.72 |         26.23 |      119545.11 |              91.91
  300 |        2249.99 |         29.23 |      385181.77 |             170.85


## 6. Karatsuba Algorithm

In [8]:
# 1. Conventional Large Integer Multiplication O(n^2)
def mult_conventional(x, y):
    s_y = str(y)
    total = 0
    for i, digit in enumerate(reversed(s_y)):
        total += (x * int(digit)) * (10 ** i)
    return total

# 2. Karatsuba Divide and Conquer Multiplication O(n^1.585)
def karatsuba(x, y):
    if x < 10 or y < 10:
        return x * y
    n = max(len(str(x)), len(str(y)))
    if n <= 16:
        return x * y
    m = n // 2
    p = 10 ** m
    x1, x0 = divmod(x, p)
    y1, y0 = divmod(y, p)
    z0 = karatsuba(x0, y0)
    z2 = karatsuba(x1, y1)
    z1 = karatsuba(x0 + x1, y0 + y1) - z0 - z2
    return z2 * (10 ** (2 * m)) + z1 * p + z0


In [11]:
random.seed(42)
print(f"{'Digits':>6} | {'Conventional (µs)':>18} | {'Karatsuba (µs)':>16} | {'Speedup':>10} | {'Matches':>8}")
print("-" * 68)

for d in [32, 64, 128, 256, 512, 1024]:
    x = random.randint(10 ** (d - 1), 10 ** d - 1)
    y = random.randint(10 ** (d - 1), 10 ** d - 1)
    # print(f"X: {x}")
    # print(f"X: {y}")
    (res_conv, t_conv) = timer(mult_conventional, x, y)
    (res_kar, t_kar) = timer(karatsuba, x, y)
    
    match = (res_conv == res_kar == (x * y))
    speedup = t_conv / t_kar if t_kar > 0 else 1.0
    print(f"{d:>6} | {t_conv:>18.2f} | {t_kar:>16.2f} | {speedup:>9.2f}x | {str(match):>8}")


Digits |  Conventional (µs) |   Karatsuba (µs) |    Speedup |  Matches
--------------------------------------------------------------------
    32 |              67.98 |            11.97 |      5.68x |     True
    64 |              53.39 |            42.20 |      1.27x |     True
   128 |             148.44 |           119.62 |      1.24x |     True
   256 |             492.73 |           280.33 |      1.76x |     True
   512 |            2908.68 |          2554.16 |      1.14x |     True
  1024 |            9207.84 |          2625.18 |      3.51x |     True
